In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math, display
sp.init_printing()

In [2]:
def fancy_display(before_text, latex, after_text=r'', latexer=sp.latex, mather=Math):
    """
    For example, 
    fancy_display(r'E = ', E, r',')
    """
    if mather is None:
        mather  = lambda i: i
    if latexer is None:
        latexer  = lambda i: i
    display(mather(before_text + latexer(latex) + after_text))

## Задание 1

> Решить задачу № 719: найти вторую квадратичную форму однополостного гиперболоида вращения
> $$x = a \cosh u \cos v, y = a \cosh u \sin v, z = c \sinh u.$$


In [3]:
u, v, t = sp.symbols('u,v,t', real=True)
a, b, c = sp.symbols('a,b,c', real=True)

In [4]:
r = sp.Matrix([
    a * sp.cosh(u) * sp.cos(v),
    a * sp.cosh(u) * sp.sin(v),
    c * sp.sinh(u)
])

fancy_display(r'r = ', r)

<IPython.core.display.Math object>

In [76]:
def get_normal(r):
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = r_u.cross(r_v)
    n /= n.norm()
    return n

def get_I_and_II(r):
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = get_normal(r)
    
    r_uu = r_u.diff(u)
    r_uv = r_u.diff(v)
    r_vv = r_v.diff(v)

    E = r_u.dot(r_u).simplify()
    F = r_u.dot(r_v).simplify()
    G = r_v.dot(r_v).simplify()
    
    L = r_uu.dot(n).simplify()
    M = r_uv.dot(n).simplify()
    N = r_vv.dot(n).simplify()

    I  = (E, F, G)
    II = (L, M, N)

    return I, II

In [77]:
_, (L, M, N) = get_I_and_II(r)

In [78]:
fancy_display(r'\Pi = L\, \mathrm{d}u^2 + 2 M\, \mathrm{d}u \mathrm{d}v + N\, \mathrm{d}v^2, \text{где}', '')
fancy_display(r'L = ', L, r',')
fancy_display(r'M = ', M, r',')
fancy_display(r'N = ', N, r'.')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Видно, что знаменатели немного ещё можно упростить, сократить на $a$ и числитель, и знаменатель и много чего в жизни ещё сделать, но я не буду

## Задание 2

> Решить задачу № 734: доказать, что главные направления прямого геликоида делят пополам углы между направлениями его образующей и винтовой линии.

In [11]:
u, v, t = sp.symbols('u,v,t', real=True)
a, b, c = sp.symbols('a:c', real=True, positive=True)
k       = sp.Symbol('k', real=True)
x1, x2  = sp.symbols('x_1:3', real=True)

In [66]:
r = sp.Matrix([
    u * sp.cos(v),
    u * sp.sin(v),
    a * v
])

fancy_display(r'r = ', r)

<IPython.core.display.Math object>

In [67]:
(E, F, G), (L, M , N) = get_I_and_II(r)

L, M, N

In [68]:
def get_main_directions(r, normalize=True):
    """
    Вычисляет главные направления и главные кривизны 
    для заданной поверхности r
    """

    k = sp.symbols('k')
    (E, F, G), (L, M , N) = get_I_and_II(r)
    
    A = sp.Matrix([[E, F], [F, G]])
    B = sp.Matrix([[L, M], [M, N]])
    
    C = B - k * A
    equation_k = sp.expand(C.det())
    
    roots = sp.solve(equation_k, k)
    
    # Обработка случая омбилических точек (когда k1 == k2)
    if len(roots) == 1:
        k1 = k2 = roots[0]
        # Любые два ортогональных направления являются главными
        k1_dir = sp.Matrix([1, 0])
        k2_dir = sp.Matrix([0, 1])
    else:
        k1, k2 = roots[0], roots[1]
        C1 = sp.simplify(C.subs(k, k1))
        C2 = sp.simplify(C.subs(k, k2))
        k1_dir = sp.simplify(C1.nullspace()[0])
        k2_dir = sp.simplify(C2.nullspace()[0])

    if normalize:
        # Нормировка с учётом метрики поверхности (первой квадратичной формы A)
        norm1 = sp.sqrt((k1_dir.T * A * k1_dir)[0])
        norm2 = sp.sqrt((k2_dir.T * A * k2_dir)[0])
        k1_dir = sp.simplify(k1_dir / norm1)
        k2_dir = sp.simplify(k2_dir / norm2)
    
    return k1_dir, k2_dir, (k1, k2)

In [74]:
u0, v0 = sp.symbols('u_0,v_0', real=True) # точка, через которую будет проходить образующая

r_u = r.diff(u)
r_v = r.diff(v)

P = {u: u0, v: v0}

generatrix = r_u.subs(P) # направляющий вектор образующей
generatrix = sp.simplify(generatrix / generatrix.norm())

d = r_v.subs(P) # направление винтовой линиии
d = sp.simplify(d / d.norm())


r1_2d, r2_2d, _ = get_main_directions(r, normalize=False) # главные направления
r1 = (r1_2d[0] * r_u + r1_2d[1] * r_v).subs(P)
r1 = sp.simplify(r1 / r1.norm())
r2 = (r2_2d[0] * r_u + r2_2d[1] * r_v).subs(P)
r2 = sp.simplify(r2 / r2.norm())

cosA = sp.trigsimp(generatrix.dot(d))
cosB1 = sp.trigsimp(d.dot(r1))
cosB2 = sp.trigsimp(d.dot(r2))
cosC1 = sp.trigsimp(generatrix.dot(r1))
cosC2 = sp.trigsimp(generatrix.dot(r2))

fancy_display(r'\cos \angle (\text{образ.}, \text{винт. линия}) = ', cosA)
fancy_display(r'\cos \angle (r_1, \text{винт. линия}) = ', cosB1)
fancy_display(r'\cos \angle (r_2, \text{винт. линия}) = ', cosB2)
fancy_display(r'\cos \angle (r_1, \text{образ.}) = ', cosC1)
fancy_display(r'\cos \angle (r_2, \text{образ.}) = ', cosC2)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Образующая ортогональна винтовой линии, а углы между ними и главными направлениями равны по $45\degree$. Что и требовалось доказать.

## Задание 3

> Решить задачу № 739. На поверхности $$x = u^2 + v^2, y = u^2 - v^2, z = u v$$ дана точка $P$, соответствующая $u = 1, v = 1$.
>  + Вычислить главные кривизны поверхности в точке $P$.
>  + Найти главные направления в этой точке и касательные к соответствующим главным нормальным сечениям.
>  + Вычислить нормальную кривизну в направлении, касательном к линии $v = u^2$.


In [79]:
a = sp.Symbol('a', positive=True, real=True)
r = sp.Matrix([u**2 + v**2, u**2 - v**2, u * v])

P = {u: 1, v: 1}
rP = r.subs(P)

fancy_display(r'r = ', r, r'.')

<IPython.core.display.Math object>

In [91]:
def k_n(r, rho, p):
    """
    rho - касательное направление (функция от аргумента p)
    """
    
    (E, F, G), (L, M , N) = get_I_and_II(r)
    x = rho.diff(p)

    A = sp.Matrix([[E, F], [F, G]])
    B = sp.Matrix([[L, M], [M, N]])

    I_x  = (x.T * A * x)[0]
    II_x = (x.T * B * x)[0]

    k_n = II_x / I_x

    return k_n

In [97]:
k1_2d, k2_2d, (k1, k2) = get_main_directions(r)

r_u = r.diff(u)
r_v = r.diff(v)

k1_2d = k1_2d.subs(P)
k2_2d = k2_2d.subs(P)

r1 = (r1_2d[0] * r_u + r1_2d[1] * r_v).subs(P)
r2 = (r2_2d[0] * r_u + r2_2d[1] * r_v).subs(P)

parametrization = sp.Matrix([t, t**2])

k_n_P = k_n(r, parametrization, t).subs(P).subs(t, 1)

fancy_display(r'\text{Глав. кривизны} = ', (k1, k2))
fancy_display(r'\text{Глав. направления} = ', (k1_2d, k2_2d))
fancy_display(r'k_n = ', k_n_P)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Задание 4

> Пусть на связной регулярной поверхности вторая квадратичная форма тождественно равна нулю: $$\Pi \equiv 0.$$ Докажите, что нормальный вектор поверхности постоянен и поверхность содержится в некоторой плоскости.

Не знаю, спать пойду.